In [ ]:
!pip install -q \
    transformers \
    peft \
    trl \
    datasets \
    faiss-cpu \
    sentence-transformers \
    tqdm \
    torch \
    bitsandbytes \
    accelerate \
    paddleocr \
    pdf2image \
    paddlepaddle \
    pillow \
    pyngrok \
    nest_asyncio \
    uvicorn \
    fastapi \
    python-multipart \
    dotenv \
    httpx

!apt-get update && apt-get install -y poppler-utils


In [36]:
%%writefile utils.py
import os
import torch
import numpy as np
import faiss
import re
import gc

def reset_memory():
    gc.collect()

def clean_text(t):
    """Clean and normalize the input text."""
    return re.sub(r'\s+', ' ', t.strip())

def save(embs, docs, embedding_path, document_path):
    """Save embeddings and documents to disk."""
    np.save(embedding_path, embs.cpu().numpy())
    with open(document_path, "w", encoding="utf-8") as f:
        f.writelines(f"{doc}\n" for doc in docs)

def load(embedding_path, document_path):
    """Load embeddings and documents from disk."""
    if not os.path.exists(embedding_path) or not os.path.exists(document_path):
        return None, None
    embs = torch.tensor(np.load(embedding_path))
    with open(document_path, "r", encoding="utf-8") as f:
        docs = f.read().splitlines()
    return embs, docs

def save_index(index, faiss_index_path):
    """Save FAISS index to disk."""
    faiss.write_index(index, faiss_index_path)

def load_index(faiss_index_path):
    """Load FAISS index from disk."""
    return faiss.read_index(faiss_index_path)

def build_index(embs):
    """Build a FAISS index from embeddings."""
    embs = embs.cpu().numpy().astype("float32")
    faiss.normalize_L2(embs)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    return index

def clean_and_overwrite_answer_file(file_path):
    """Clean and format the answers in the provided file."""
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    qa_blocks = re.findall(r"Query: (.*?)\n+Answer: (.*?)(?=\n+Query:|\Z)", content, re.DOTALL)
    cleaned_output = ""
    for query, answer in qa_blocks:
        cleaned_output += f"Question: {query.strip()}\nAnswer: {answer.strip()}\n\n"
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(cleaned_output.strip())
    print(f"Answers cleaned and saved to: {file_path}")

def load_user_docs(user_file):
    """Load and clean documents from a file."""
    if os.path.exists(user_file):
        with open(user_file, "r", encoding="utf-8") as file:
            user_docs = file.readlines()
        return [clean_text(doc) for doc in user_docs]
    return []

def truncate_to_last_complete_sentence(text):
    # This splits on punctuation followed by any whitespace (space, newline, tab, etc.)
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())

    # If text does not end with punctuation, remove the last sentence (likely incomplete)
    if len(sentences) > 1 and not text.strip()[-1] in '.!?':
        sentences = sentences[:-1]

    # Join sentences back
    return ' '.join(sentences).strip()


Overwriting utils.py


In [37]:
%%writefile models.py
import torch
import re
from transformers import (
	AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
	AutoModelForCausalLM, AutoModelForSeq2SeqLM, BitsAndBytesConfig
)
from utils import truncate_to_last_complete_sentence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bnb_config = BitsAndBytesConfig(
	load_in_4bit=True,
	bnb_4bit_compute_dtype=torch.float16,
	bnb_4bit_use_double_quant=True,
	bnb_4bit_quant_type="nf4"
)

def load_encoder(name, quantized=False):
	tokenizer = AutoTokenizer.from_pretrained(name)
	if quantized:
		model = AutoModel.from_pretrained(name, device_map="auto", quantization_config=bnb_config)
	else:
		model = AutoModel.from_pretrained(name).to(device)
	return tokenizer, model

def load_reranker(name, quantized=False):
	tokenizer = AutoTokenizer.from_pretrained(name)
	if quantized:
		model = AutoModelForSequenceClassification.from_pretrained(name, device_map="auto", quantization_config=bnb_config)
	else:
		model = AutoModelForSequenceClassification.from_pretrained(name).to(device)
	return tokenizer, model

def load_summarizer(name, quantized=False):
	tokenizer = AutoTokenizer.from_pretrained(name)
	if quantized:
		model = AutoModelForSeq2SeqLM.from_pretrained(name, device_map="auto", quantization_config=bnb_config)
	else:
		model = AutoModelForSeq2SeqLM.from_pretrained(name).to(device)
	return tokenizer, model

def load_generator(model_name, quantized=True):
	tokenizer = AutoTokenizer.from_pretrained(model_name)
	tokenizer.pad_token = tokenizer.eos_token
	if quantized:
		model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", quantization_config=bnb_config)
	else:
		model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
	return tokenizer, model

def encode_query(query, tokenizer, model, max_query_length):
	inputs = tokenizer(query, return_tensors="pt", padding=True, truncation=True, max_length=max_query_length).to(device)
	with torch.no_grad():
		embeddings = model.base_model(**inputs).last_hidden_state.mean(dim=1)
	return embeddings

def rerank(query, candidates, tokenizer, model):
	inputs = [tokenizer(query, doc, return_tensors="pt", padding=True, truncation=True).to(device) for doc in candidates]
	scores = [model(**input).logits[0].item() for input in inputs]
	doc_scores = list(zip(candidates, scores))
	doc_scores.sort(key=lambda x: x[1], reverse=True)

	return doc_scores

def summarize(text, tokenizer, model, max_input_len=512, max_output_len=150):
	inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_input_len).to(device)
	summary_ids = model.generate(
		inputs["input_ids"],
		max_length=max_output_len,
		num_beams=4,
		early_stopping=True
	)
	return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

def generate_answer(query, wiki_context, user_context, tokenizer, model, max_new_tokens, temperature, top_p):
    input_text = f"Question: {query}\n"
    input_text += f"Most relevant information: {user_context}\n"
    input_text += f"Additional reference (Wikipedia): {wiki_context}\n"
    input_text += "Answer:"

    print(input_text)  # Optional: helpful for debugging

    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True).to(model.device)
    model.config.pad_token_id = model.config.eos_token_id

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
				early_stopping=True
    )

    # Decode the full output
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract answer portion after "Answer:"
    if "Answer:" in decoded:
        answer = decoded.split("Answer:")[1].strip()
    else:
        answer = decoded.strip()

    # Post-process: truncate any incomplete final sentence
    answer = truncate_to_last_complete_sentence(answer)

    return answer


Overwriting models.py


In [38]:
%%writefile ocr_pipeline.py
import os
from pdf2image import convert_from_path
from paddleocr import PaddleOCR
from models import summarize
import re

# === Init OCR model ===
ocr = PaddleOCR(use_angle_cls=True, lang='en')

# === PDF to Image ===
def pdf_to_images(pdf_path, dpi=300):
	return convert_from_path(pdf_path, dpi)

def save_images(pages, out_dir):
	os.makedirs(out_dir, exist_ok=True)
	paths = []
	for i, page in enumerate(pages):
		img_path = os.path.join(out_dir, f"page_{i}.jpg")
		page.save(img_path, 'JPEG')
		paths.append(img_path)
	return paths

# === OCR Text Extraction ===
def extract_text_with_paddleocr(img_path):
	result = ocr.ocr(img_path, cls=True)
	lines = [line[1][0] for line in result[0]]
	return "\n".join(lines)

# === Cleaning & Chunking ===
def clean_text(text):
	return re.sub(r'\s+', ' ', text.strip())

def chunk_text(text, max_words=300):
	words = text.split()
	return [" ".join(words[i:i + max_words]) for i in range(0, len(words), max_words)]

def process_uploaded_files(
	file_paths,
	output_txt=None,
	output_pages=None,
	summarizer_tokenizer=None,
	summarizer_model=None
):

	all_docs = []

	if output_txt is not None:
		os.makedirs(os.path.dirname(output_txt), exist_ok=True)

	for filepath in file_paths:
		filename = os.path.basename(filepath)
		print(f"\nProcessing: {filepath}")

		if filename.lower().endswith(".pdf"):
			pages = pdf_to_images(filepath)
			img_dir = os.path.join(output_pages or "output_pages", os.path.splitext(filename)[0])
			image_paths = save_images(pages, out_dir=img_dir)
		elif filename.lower().endswith((".png", ".jpg", ".jpeg")):
			image_paths = [filepath]
		else:
			print(f"Skipping unsupported file: {filename}")
			continue

		all_text = ""
		for img_path in image_paths:
			all_text += clean_text(extract_text_with_paddleocr(img_path))

		lines = chunk_text(all_text)

		# Always summarize
		summarized_lines = []
		for i, line in enumerate(lines):
			print(f"Summarizing chunk {i+1}/{len(lines)}...")
			summarized_line = summarize(line, summarizer_tokenizer, summarizer_model)
			summarized_lines.append(summarized_line)
		lines = summarized_lines

		all_docs.extend(lines)

		if output_txt:
			with open(output_txt, "a", encoding="utf-8") as f:
				for line in lines:
					f.write(line + "\n")

	if output_txt:
		print(f"\n OCR complete. Output saved to: {output_txt}")

	return all_docs


Overwriting ocr_pipeline.py


In [39]:
%%writefile main.py
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from pathlib import Path
import shutil
import os
import torch
import numpy as np
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from contextlib import asynccontextmanager
import asyncio
import re

os.environ["TOKENIZERS_PARALLELISM"] = "false"

if torch.cuda.is_available():
    print("CUDA is available!")
    print(f"Using device: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available.")

from ocr_pipeline import process_uploaded_files, chunk_text
from utils import (
    clean_text, build_index, save_index, load_index,
    load_user_docs, clean_and_overwrite_answer_file, reset_memory
)
from models import (
    load_encoder, load_reranker, load_generator, load_summarizer,
    encode_query, summarize, rerank, generate_answer, device
)

BASE_DIR = "."

documents_and_index = os.path.join(BASE_DIR, "documents_and_index")
embedding_path = os.path.join(documents_and_index, "embeddings.npy")
document_path = os.path.join(documents_and_index, "documents.txt")
faiss_index_path = os.path.join(documents_and_index, "faiss_index.index")
user_docs_path = os.path.join(documents_and_index, "user_docs.txt")
answer_path = os.path.join(BASE_DIR, "answer.txt")
input_folder = os.path.join(BASE_DIR, "uploaded_files")
output_pages = os.path.join(BASE_DIR, "output_pages")
grok_url_file = os.path.join(BASE_DIR, "grok_url.txt")

top_k = 5
docs_to_embed = 1000
batch_size = 8
max_query_length = 512
max_new_tokens = 100
relevance_threshold = -3
temperature = 0.7
top_p = 0.9

encoder_model_name = "BAAI/bge-base-en-v1.5"
reranker_model_name = "BAAI/bge-reranker-large"
generator_model_name = "deepcogito/cogito-v1-preview-llama-3B"
summarizer_model_name = "facebook/bart-large-cnn"

# Globals initialized on startup
gen_tok = gen_model = None
enc_tok = enc_model = None
rr_tok = rr_model = None
sum_tok = sum_model = None
summarizer_tokenizer, summarizer_model = load_summarizer(summarizer_model_name)
embs = None  # torch.Tensor
docs = None  # list of str
index = None
user_docs = []

processed_files = set()
public_url = None

async def launch_grok_and_capture_url():
    """
    Launches `grok http 8000` as a subprocess,
    reads its stdout lines asynchronously,
    extracts the public URL, writes it to grok_url.txt,
    and returns the URL.
    """
    global public_url

    # Make sure grok command is available on your system path
    proc = await asyncio.create_subprocess_exec(
        "grok", "http", "8000",
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.STDOUT,
        text=True
    )

    url_pattern = re.compile(r"https://[a-zA-Z0-9.-]+\.grok\.dev")

    while True:
        line = await proc.stdout.readline()
        if not line:
            break
        print(f"[grok] {line.strip()}")
        match = url_pattern.search(line)
        if match:
            public_url = match.group(0)
            print(f"Detected Grok public URL: {public_url}")
            with open(grok_url_file, "w", encoding="utf-8") as f:
                f.write(public_url)
            # Optionally: break after first URL detected
            break

    # Note: proc keeps running (grok tunnels traffic)
    return public_url

@asynccontextmanager
async def lifespan(app: FastAPI):
    global gen_tok, gen_model
    global enc_tok, enc_model
    global rr_tok, rr_model
    global sum_tok, sum_model
    global embs, docs, index
    global user_docs
    global processed_files

    os.makedirs(input_folder, exist_ok=True)
    os.makedirs(output_pages, exist_ok=True)
    os.makedirs(documents_and_index, exist_ok=True)

    print("Loading models...")
    gen_tok, gen_model = load_generator(generator_model_name)
    enc_tok, enc_model = load_encoder(encoder_model_name)
    rr_tok, rr_model = load_reranker(reranker_model_name)
    sum_tok = AutoTokenizer.from_pretrained(summarizer_model_name)
    sum_model = AutoModelForSeq2SeqLM.from_pretrained(summarizer_model_name).to(device)

    # Load embeddings and documents if they exist
    if os.path.exists(embedding_path) and os.path.exists(document_path):
        print("Loading embeddings and documents from disk...")
        embs_np = np.load(embedding_path)
        embs = torch.tensor(embs_np)
        with open(document_path, "r", encoding="utf-8") as f:
            docs = [line.strip() for line in f.readlines() if line.strip()]
        print(f"Loaded {len(docs)} documents and embeddings.")
    else:
        print("Embeddings or docs missing, building from Wikipedia dataset...")
        wiki = load_dataset("wikipedia", "20220301.en", split=f"train[:{docs_to_embed}]", trust_remote_code=True)

        docs = []
        all_embs = []
        max_tokens = 512
        stride = 256

        print("Embedding Wikipedia documents...")
        for ex in tqdm(wiki, desc="Processing Wikipedia docs"):
            raw_text = clean_text(ex["text"])

            # Word-level chunking (max 100 words)
            word_chunks = chunk_text(raw_text, max_words=100)

            for chunk_text_piece in word_chunks:
                tokens = enc_tok.encode(chunk_text_piece, add_special_tokens=False)
                for i in range(0, len(tokens), stride):
                    chunk_tokens = tokens[i:i + max_tokens]
                    if len(chunk_tokens) < 10:
                        continue

                    chunk_tokens_tensor = torch.tensor([chunk_tokens]).to(device)
                    attention_mask = torch.ones_like(chunk_tokens_tensor).to(device)

                    with torch.no_grad():
                        outputs = enc_model(input_ids=chunk_tokens_tensor, attention_mask=attention_mask)
                        emb = outputs.last_hidden_state.mean(dim=1).cpu()

                    all_embs.append(emb)
                    chunk_decoded_text = enc_tok.decode(chunk_tokens, skip_special_tokens=True)
                    docs.append(chunk_decoded_text)

                    reset_memory()

        embs = torch.cat(all_embs, dim=0)
        # Save embeddings as numpy
        np.save(embedding_path, embs.numpy())
        with open(document_path, "w", encoding="utf-8") as f:
            f.write("\n".join(docs))

    # Load or build FAISS index
    if os.path.exists(faiss_index_path):
        print("Loading FAISS index...")
        index = load_index(faiss_index_path)
    else:
        print("Building FAISS index...")
        index = build_index(embs)
        save_index(index, faiss_index_path)

    # Load user docs from disk
    user_docs = load_user_docs(user_docs_path)
    print(f"Loaded {len(user_docs)} user docs")

    # Track processed files to avoid re-processing
    processed_files = set(os.listdir(input_folder))

    yield  # app runs here

app = FastAPI(lifespan=lifespan)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class QueryRequest(BaseModel):
    query: str

@app.post("/upload/")
async def upload_file(file: UploadFile = File(...)):
    global processed_files
    global user_docs

    if file.filename in processed_files:
        return {"message": f"File '{file.filename}' already processed."}

    save_path = Path(input_folder) / file.filename
    with save_path.open("wb") as buffer:
        shutil.copyfileobj(file.file, buffer)
    processed_files.add(file.filename)

    # Process only the newly uploaded file
    user_docs = process_uploaded_files(
        file_paths=[str(save_path)],
        output_txt=user_docs_path,
        output_pages=output_pages,
        summarizer_tokenizer=summarizer_tokenizer,
        summarizer_model=summarizer_model,
    )

    return {"message": f"Processed file '{file.filename}'"}

@app.post("/query/")
def answer_query(req: QueryRequest):
    query = req.query

    global user_docs
    if not user_docs:
        user_docs = load_user_docs(user_docs_path)


    # --- Step 1: Get Userdocs ---

    user_reranked = rerank(query, user_docs, rr_tok, rr_model)  # returns list of (doc, score)
    user_reranked_relevant = [(doc, score) for doc, score in user_reranked if score > relevance_threshold]
    user_reranked_relevant.sort(key=lambda x: x[1], reverse=True) # sort by score

    chosen_user_docs = [doc for doc, _ in user_reranked_relevant] # remove score from each tuple

    print("Step 1 complete: Userdocs retrieved")


    # --- Step 2: Pad with relevant Wikipedia docs if needed ---

    num_wiki_docs = max(0, top_k - len(chosen_user_docs))
    chosen_wiki_docs = []

    if num_wiki_docs > 0:
        query_emb = encode_query(query, enc_tok, enc_model, max_query_length)
        _, top_idx = index.search(query_emb.cpu().numpy(), top_k)  # retrieve more for reranking quality
        wiki_docs = [docs[i] for i in top_idx[0]]

        wiki_reranked = rerank(query, wiki_docs, rr_tok, rr_model)  # returns list of (doc, score)
        wiki_reranked_relevant = [(doc, score) for doc, score in wiki_reranked if score > relevance_threshold]
        wiki_reranked_relevant.sort(key=lambda x: x[1], reverse=True)

        chosen_wiki_docs_raw = [doc for doc, _ in wiki_reranked[:num_wiki_docs]]
        chosen_wiki_docs = [summarize(doc, summarizer_tokenizer, summarizer_model) for doc in chosen_wiki_docs_raw]

    print("Step 2 complete: Wikidocs retrieved")


    # --- Step 3: LLM Query ---

    user_context = " ".join(chosen_user_docs)
    wiki_context = " ".join(chosen_wiki_docs)

    if not chosen_user_docs and not chosen_wiki_docs:
        return JSONResponse(content={"answer": "Sorry, I couldn't find any relevant information to answer that."})

    answer = generate_answer(query, wiki_context, user_context, gen_tok, gen_model, max_new_tokens, temperature, top_p)

    with open(answer_path, "w", encoding="utf-8") as f:
        f.write(f"Query: {query}\n\nAnswer: {answer}\n\n")

    clean_and_overwrite_answer_file(answer_path)

    return JSONResponse(content={"answer": answer})

@app.get("/status/")
def get_status():
    return {"status": "ok"}


Overwriting main.py


In [40]:
import os
import nest_asyncio
import uvicorn
from pyngrok import ngrok
import main
import requests

# Try to import Colab userdata to get secrets if running in Colab
try:
    from google.colab import userdata
except ImportError:
    userdata = None

def get_secret(name):
    # Try Colab userdata first (if available), then environment variables
    if userdata:
        val = userdata.get(name)
        if val:
            return val
    return os.environ.get(name)

# Apply asyncio fix for Colab
nest_asyncio.apply()

# Get secrets
auth_token = get_secret("NGROK_AUTH_TOKEN")
jsonbin_url = get_secret("JSONBIN_URL")
jsonbin_api_key = get_secret("JSONBIN_API_KEY")

# Safety check for ngrok token
if not auth_token:
    raise ValueError("Missing NGROK_AUTH_TOKEN. Set it in Colab Secrets tab or environment variables.")

# Setup ngrok
ngrok.set_auth_token(auth_token)
ngrok.kill()
public_url = ngrok.connect(8000, bind_tls=True).public_url
print(f"Public URL: {public_url}")

# Let main module know the public URL
main.public_url = public_url

# Save URL to a file if needed
with open("ngrok_url.txt", "w") as f:
    f.write(public_url)

# Update JSONBin
def update_jsonbin_url(new_url: str):
    if not jsonbin_api_key or not jsonbin_url:
        print("JSONBIN_API_KEY or JSONBIN_URL not set, skipping update.")
        return

    headers = {
        "Content-Type": "application/json",
        "X-Master-Key": jsonbin_api_key,
    }
    data = {
        "url": new_url
    }
    response = requests.put(jsonbin_url, json=data, headers=headers)
    if response.status_code == 200:
        print("JSONBin updated successfully!")
    else:
        print(f"Failed to update JSONBin: {response.status_code} - {response.text}")

update_jsonbin_url(public_url)

# Run FastAPI app
uvicorn.run(main.app, host="0.0.0.0", port=8000)


Public URL: https://8ee2-34-124-227-11.ngrok-free.app
JSONBin updated successfully!


INFO:     Started server process [426]
INFO:     Waiting for application startup.


Loading models...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-34' coro=<Server.serve() done, defined at /usr/local/lib/python3.11/dist-packages/uvicorn/server.py:68> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/uvicorn/main.py", line 580, in run
    server.run()
  File "/usr/local/lib/python3.11/dist-packages/uvicorn/server.py", line 66, in run
    return asyncio.run(self.serve(sockets=sockets))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/nest_asyncio.py", line 30, in run
    return loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/nest_asyncio.py", line 92, in run_until_complete
    self._run_once()
  File "/usr/local/lib/python3.11/dist-packages/nest_asyncio.py", line 133, in _run_once
    handle._run()
  File "/usr/lib/python3.11/asyncio/events.py", line 84, in _run
    s

...

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Step 1 complete: Userdocs retrieved
Step 2 complete: Wikidocs retrieved
Question: Why did Beter Lourenco want to kill someone?
Most relevant information: Beter Lourenco was arrested for domestic terrorism, four counts of murder, and prolonged illegal drug use. The ricin attacked his nerves and he collapsed. The rest of the federal agents began chasing him. He ran behind a corner and took out his next weapon: mustard gas. Coughing and collapsing ensued. Beter knew that if he wanted to get away with this murder, he would need to play his cards right and retain control of all conversations.. possible. He knew Andy's phone passcode, so from that phone, he texted himself: "wow Beter, I didnt know you could punch like that haha" Beter then hidAndy's phone under the living room couch and later showed his sister the text on his own phone. Beter Lourenco is a former AP Research student. He is accused of throwing away the bodies of his victims. Beter is now on the run from the police. He wants t

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Step 2 complete: Wikidocs retrieved
Question: Why did Beter Lourenco want to kill someone and specifically use a frying pan?
Most relevant information: Beter Lourenco was arrested for domestic terrorism, four counts of murder, and prolonged illegal drug use. The ricin attacked his nerves and he collapsed. The rest of the federal agents began chasing him. He ran behind a corner and took out his next weapon: mustard gas. Coughing and collapsing ensued. Beter Lourenco is a former AP Research student. He is accused of throwing away the bodies of his victims. Beter is now on the run from the police. He wants to start a new life in college. He has been removed from the AP Research group chat.
Additional reference (Wikipedia): names were alternately given as dahmane abd al - sattar, husband of malika el aroud, and bouraoui el - ouaer ; or 34 - year-old karim touzani and 26 - year - old kacem bakkali. the attackers claimed to be belgians originally from morocco. their passports turned out to b

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [426]
